# Tool 4: Trade Analyzer & Bot Profiler

## Purpose
Analyzes executed trades to detect predictable bot behavior patterns.
The goal is to identify "Olivia-like" informed traders — bots that
consistently trade at daily price extremes (lows and highs).

## Why This Matters
In every IMC Prosperity iteration, there have been bots with exploitable
predictable behavior. In Prosperity 3, a bot named "Olivia" bought
exactly 15 lots at the daily low and sold 15 lots at the daily high.
Detecting this pattern was worth ~8,000 SeaShells/round. The same pattern
structure has appeared in Prosperity 1, 2, and 3, so it's very likely
present in Prosperity 4 as well.

## What You'll See
1. **Quantity distribution**: What trade sizes appear? Unusual fixed sizes
   (e.g., always 15) are a strong indicator of a specific bot.
2. **Trades at daily extremes**: Trades plotted against the running daily
   min/max. Bots that cluster at extremes are informed traders.
3. **Directional analysis**: Buy vs sell trades relative to fair price,
   grouped by quantity — isolates which size belongs to which bot.
4. **Temporal patterns**: When in the day trades of each size occur.
5. **Quantity filter view**: Isolate trades of a specific size.

## How to Hunt for Informed Traders
1. Look at the quantity histogram — any unusually common fixed sizes?
2. Filter to that quantity and see if those trades cluster at daily min/max
3. If yes, that's your informed trader signal
4. Design your algo to detect these trades in real-time and follow them

## Dependencies
- `data_loader.py`, `matplotlib`, `pandas`, `numpy`

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

PRODUCT = "ASH_COATED_OSMIUM"  # or "INTARIAN_PEPPER_ROOT"
DAY = -1

# Quantity filter: set to a specific int to isolate a bot,
# or None to show all trades
QUANTITY_FILTER = None  # e.g., 15 to isolate Olivia-like trades

FIG_WIDTH = 18
FIG_HEIGHT = 5

In [ ]:
# ============================================================
# SETUP
# ============================================================
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from data_loader import (
    load_prices, load_trades, compute_wall_mid,
    merge_trades_with_prices, AVAILABLE_DAYS, PRODUCTS,
)

prices = compute_wall_mid(load_prices(day=DAY, product=PRODUCT))
trades = load_trades(day=DAY, product=PRODUCT)
merged = merge_trades_with_prices(prices, trades)

print(f"Loaded {len(trades)} trades for {PRODUCT} on Day {DAY}")
print(f"Quantity range: {trades['quantity'].min()} — {trades['quantity'].max()}")
print(f"Unique quantities: {sorted(trades['quantity'].unique())}")

In [ ]:
# ============================================================
# PLOT 1: Trade Quantity Distribution
# ============================================================
# Look for spikes at specific quantities — those are likely individual
# bots with fixed trade sizes. In Prosperity 3, Olivia always traded
# exactly 15 lots.

fig, axes = plt.subplots(1, 2, figsize=(FIG_WIDTH, FIG_HEIGHT))

# Histogram
qty_counts = trades["quantity"].value_counts().sort_index()
axes[0].bar(qty_counts.index, qty_counts.values, color="tab:blue", alpha=0.7)
axes[0].set_xlabel("Trade Quantity")
axes[0].set_ylabel("Count")
axes[0].set_title("Trade Quantity Distribution", fontweight="bold")
axes[0].grid(True, alpha=0.3, axis="y")

# Table of top quantities
top_qty = trades["quantity"].value_counts().head(10)
axes[1].axis("off")
table_data = [[q, c, f"{c/len(trades)*100:.1f}%"] for q, c in top_qty.items()]
table = axes[1].table(
    cellText=table_data,
    colLabels=["Quantity", "Count", "% of Trades"],
    loc="center", cellLoc="center",
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.5)
axes[1].set_title("Top 10 Trade Sizes", fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# PLOT 2: Trades vs Daily Running Min/Max
# ============================================================
# The running min and max of the Wall Mid represent the daily price
# extremes so far. An informed trader (like Olivia) buys at the
# running min and sells at the running max.
# Look for: trades clustering exactly at the min/max lines.

running_min = prices["wall_mid"].expanding().min()
running_max = prices["wall_mid"].expanding().max()

fig, ax = plt.subplots(figsize=(FIG_WIDTH, FIG_HEIGHT + 2))

ts = prices["timestamp"].values
ax.plot(ts, prices["wall_mid"], color="gray", linewidth=0.8, alpha=0.5, label="Wall Mid")
ax.plot(ts, running_min, color="blue", linewidth=1, linestyle="--", alpha=0.7, label="Running Min")
ax.plot(ts, running_max, color="red", linewidth=1, linestyle="--", alpha=0.7, label="Running Max")

display_trades = trades if QUANTITY_FILTER is None else trades[trades["quantity"] == QUANTITY_FILTER]
filter_label = f" (qty={QUANTITY_FILTER})" if QUANTITY_FILTER else ""

ax.scatter(
    display_trades["timestamp"], display_trades["price"],
    s=display_trades["quantity"] * 8, c="tab:green", marker="^",
    alpha=0.8, edgecolors="black", linewidths=0.5,
    label=f"Trades{filter_label}", zorder=5,
)

ax.set_xlabel("Timestamp", fontsize=12)
ax.set_ylabel("Price", fontsize=12)
ax.set_title(f"Trades vs Daily Extremes — {PRODUCT} (Day {DAY}){filter_label}", fontsize=14, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# PLOT 3: Trade Edge Analysis by Quantity
# ============================================================
# For each trade, compute its "edge" = trade_price - wall_mid.
# Positive edge = bought above fair (taker buying).
# Negative edge = bought below fair (taker selling).
# Group by quantity to see if specific sizes have directional bias.

if "wall_mid" in merged.columns:
    merged["edge"] = merged["price"] - merged["wall_mid"]
else:
    merged["edge"] = merged["price"] - merged["mid_price"]

fig, axes = plt.subplots(1, 2, figsize=(FIG_WIDTH, FIG_HEIGHT))

# Scatter: edge vs timestamp, colored by quantity
sc = axes[0].scatter(
    merged["timestamp"], merged["edge"],
    s=merged["quantity"] * 5, c=merged["quantity"],
    cmap="viridis", alpha=0.6, edgecolors="black", linewidths=0.3,
)
axes[0].axhline(0, color="orange", linewidth=1, linestyle="--")
axes[0].set_xlabel("Timestamp")
axes[0].set_ylabel("Trade Price − Wall Mid")
axes[0].set_title("Trade Edge Over Time (colored by qty)", fontweight="bold")
axes[0].grid(True, alpha=0.3)
plt.colorbar(sc, ax=axes[0], label="Quantity")

# Box plot: edge distribution by quantity
qty_groups = sorted(merged["quantity"].unique())
if len(qty_groups) <= 20:
    edge_by_qty = [merged.loc[merged["quantity"] == q, "edge"].values for q in qty_groups]
    axes[1].boxplot(edge_by_qty, labels=qty_groups)
    axes[1].axhline(0, color="orange", linewidth=1, linestyle="--")
    axes[1].set_xlabel("Trade Quantity")
    axes[1].set_ylabel("Edge (price − wall_mid)")
    axes[1].set_title("Edge Distribution by Quantity", fontweight="bold")
    axes[1].grid(True, alpha=0.3, axis="y")
else:
    axes[1].text(0.5, 0.5, f"Too many unique quantities ({len(qty_groups)}).\nUse QUANTITY_FILTER to narrow down.",
                 ha="center", va="center", transform=axes[1].transAxes)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# PLOT 4: Temporal Patterns — When Do Trades Occur?
# ============================================================
# Some bots trade at regular intervals or cluster at specific times.
# This histogram shows trade timing for each quantity group.

fig, ax = plt.subplots(figsize=(FIG_WIDTH, FIG_HEIGHT))

# Top 5 most common quantities
top_qtys = trades["quantity"].value_counts().head(5).index.tolist()
colors = ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple"]

for i, q in enumerate(top_qtys):
    subset = trades[trades["quantity"] == q]
    ax.hist(subset["timestamp"], bins=50, alpha=0.5, label=f"qty={q} (n={len(subset)})",
            color=colors[i % len(colors)])

ax.set_xlabel("Timestamp")
ax.set_ylabel("Trade Count")
ax.set_title(f"Trade Timing by Quantity — {PRODUCT} (Day {DAY})", fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CROSS-DAY CONSISTENCY CHECK
# ============================================================
# Do the same quantity patterns appear on all 3 days?
# If a specific quantity (e.g., 15) appears consistently across days
# and always at extremes, that's strong evidence of a persistent bot.

print(f"Cross-day quantity analysis for {PRODUCT}:\n")
for d in AVAILABLE_DAYS:
    t = load_trades(day=d, product=PRODUCT)
    print(f"Day {d}: {len(t)} trades")
    print(f"  Quantities: {sorted(t['quantity'].unique())}")
    print(f"  Top 5: {t['quantity'].value_counts().head(5).to_dict()}")
    print()